In [0]:
customers_df = spark.read \
.format("parquet") \
.load("/Volumes/finance_domain_project/default/raw_data/Cleaned/customers_parquet/")

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS finance_project")

In [0]:
spark.sql("SHOW SCHEMAS").show()


In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS finance_raw_location
URL 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance/'
WITH (STORAGE CREDENTIAL finance_uc_cred);

In [0]:
# Copy from volume to ADLS Gen2
spark.read.parquet("/Volumes/finance_domain_project/default/raw_data/Cleaned/customers_parquet/") \
  .write \
  .format("parquet") \
  .mode("overwrite") \
  .save("abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance/customers_parquet/")


In [0]:
%sql
CREATE TABLE finance_project.customers_external
(member_id string, emp_title string, emp_length int, home_ownership string, 
 annual_income float, address_state string, address_zip_code string, 
 address_country string, grade string, sub_grade string, 
 verification_status string, tot_hi_cred_limit float, application_type string, 
 join_annual_income float, verification_status_joint string, ingest_date timestamp) 
USING PARQUET 
LOCATION 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance/customers_parquet/';


In [0]:
%sql
SHOW EXTERNAL LOCATIONS; 

In [0]:
%sql
SELECT * FROM finance_project.customers_external LIMIT 10;


In [0]:
%sql
DESCRIBE EXTENDED finance_project.customers_external;


In [0]:
loans_df = spark.read \
.format("parquet") \
.load("/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_parquet")


In [0]:
#Copy loans from volume to external storage
spark.read.parquet("/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_parquet/") \
  .write \
  .format("parquet") \
  .mode("overwrite") \
  .save("abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance/loans_parquet/")

In [0]:
loans_df.printSchema()  # See column names & types
display(loans_df.limit(5))  # Preview data

In [0]:
spark.sql("""
CREATE TABLE finance_project.loans_external
(loan_id string, member_id string, loan_amount float, funded_amount float,
 loan_term_years integer, interest_rate float, monthly_installlment float,
 issue_date string, loan_status string, loan_purpose string, 
 loan_title string, ingest_date timestamp) 
USING PARQUET 
LOCATION 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance/loans_parquet/'
""")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8819567140596862>, line 1
----> 1 spark.sql("""
      2 CREATE TABLE finance_project.loans_external
      3 (loan_id string, member_id string, loan_amount float, funded_amount float,
      4  loan_term_years integer, interest_rate float, monthly_installlment float,
      5  issue_date string, loan_status string, loan_purpose string, 
      6  loan_title string, ingest_date timestamp) 
      7 USING PARQUET 
      8 LOCATION 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance/loans_parquet/'
      9 """)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    898         _views.append(SubqueryAlias(df._plan, name))
    900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.client.exe

In [0]:
spark.sql("SHOW COLUMNS FROM finance_project.loans_external").show()


In [0]:
%sql SHOW TABLES IN finance_project


In [0]:
%sql
 SELECT * FROM finance_project.loans_external LIMIT 5

In [0]:
loans_repayments_df = spark.read \
  .format("parquet") \
  .load("/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_repayments_parquet/")

In [0]:
loans_repayments_df.printSchema()


In [0]:
display(loans_repayments_df.limit(5))

In [0]:
loans_repayments_df.write \
  .format("parquet") \
  .mode("overwrite") \
  .save("abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance/loans_repayments_parquet/")
  

In [0]:
%sql
CREATE TABLE finance_project.loans_repayments_external
-- (repayment_id string, loan_id string, payment_date timestamp, amount float, ...)
USING PARQUET 
LOCATION 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance/loans_repayments_parquet/';

In [0]:
spark.sql("SHOW COLUMNS FROM finance_project.loans_repayments_external").show(truncate=False)


In [0]:
del loans_defaulters_delinq_df

In [0]:
loans_defaulters_delinq_df = spark.read \
  .format("parquet") \
  .load("/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_defaulters_delinq_parquet/")


In [0]:
# Create NEW DataFrame using spark.sql (bypasses path cache)
loans_defaulters_delinq_df = spark.read.parquet("/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_defaulters_delinq_parquet/")

In [0]:
loans_defaulters_delinq_df.printSchema()


In [0]:
display(loans_defaulters_delinq_df.limit(5))


In [0]:
loans_defaulters_delinq_df.write \
  .format("parquet") \
  .mode("overwrite") \
  .save("abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance/loans_defaulters_delinq_parquet/")

In [0]:
%sql
CREATE TABLE finance_project.loans_defaulters_delinq_external
(member_id string, 
 delinq_2yrs integer, 
 delinq_amnt float, 
 mths_since_last_delinq integer)
USING PARQUET
LOCATION 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance/loans_defaulters_delinq_parquet/'


In [0]:
loans_defaulters_detail_records_enq_df = spark.read.parquet("/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_defaulters_detail_records_enq_parquet/")

In [0]:
loans_defaulters_detail_records_enq_df.printSchema()


In [0]:
display(loans_defaulters_detail_records_enq_df.limit(5))


In [0]:
loans_defaulters_detail_records_enq_df.write \
  .format("parquet") \
  .mode("overwrite") \
  .save("abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance/loans_defaulters_detail_records_enq_parquet/")

In [0]:
%sql
CREATE TABLE finance_project.loans_defaulters_detail_records_enq_external
(member_id string, 
 pub_rec integer,
 pub_rec_bankruptcies integer, 
 inq_last_6mths integer)
USING PARQUET
LOCATION 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance/loans_defaulters_detail_records_enq_parquet/' 

In [0]:
display(loans_defaulters_detail_records_enq_df)

In [0]:
%sql SHOW TABLES IN finance_project;
